# Finite-MDP policy evaluation

**Learning goals:** construct a fixed-policy transition matrix, solve its Bellman equation directly, and compare the solution with successive approximation.

**Predict first:** as $\gamma$ moves from 0.5 to 0.98, what happens to the number of iterations needed for the same tolerance?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

P_pi = np.array([[0.80, 0.20, 0.00],
                 [0.10, 0.70, 0.20],
                 [0.00, 0.25, 0.75]])
r_pi = np.array([0.0, 1.0, 2.0])

def evaluate_policy(gamma=0.90, tolerance=1e-10):
    exact = np.linalg.solve(np.eye(3) - gamma * P_pi, r_pi)
    value = np.zeros(3)
    errors = []
    for _ in range(10_000):
        updated = r_pi + gamma * P_pi @ value
        errors.append(np.max(np.abs(updated - exact)))
        value = updated
        if errors[-1] < tolerance:
            break
    return exact, value, np.asarray(errors)

exact, approximate, errors = evaluate_policy()
print("Exact value:", exact)
print("Iterations:", len(errors))

In [ ]:
@interact(gamma=FloatSlider(value=0.90, min=0.50, max=0.98, step=0.01))
def convergence_plot(gamma):
    exact, approximate, errors = evaluate_policy(gamma)
    plt.figure(figsize=(6, 3))
    plt.semilogy(errors)
    plt.xlabel("Iteration")
    plt.ylabel("Sup-norm error")
    plt.title(f"Policy evaluation, gamma={gamma:.2f}")
    plt.grid(alpha=0.25)
    plt.show()

**Experiment:** change one row of `P_pi` so the chain remains longer in state 2. Predict the sign of the change in each value before rerunning.

In [ ]:
assert np.allclose(P_pi.sum(axis=1), 1.0)
assert np.allclose(approximate, exact, atol=1e-9)
assert np.all(errors[1:] <= errors[:-1] + 1e-12)
print("Checks passed.")